# 📐 S&P 500 Pairs Trading 形成期 (Formation Period) 策略邏輯與公式詳解

## 📝 概述

在配對交易 (Pairs Trading) 中，**形成期 (Formation Period)**（預設 $F = 252$ 天）的核心任務是**篩選候選標的並建立具備統計套利價值的配對組合**。本文件涵蓋 `strategies/config.py` 現役 `strategies_raw_all` 的全部 10 個策略，以及已封存策略的精簡摘要與封存依據。

> **⚠ 注意：** 本文件以 `strategies/formation/` 下實際運行的 `.py` 原始碼為唯一依據，所有公式、閾值與欄位命名均與程式碼保持一致。已封存策略的完整診斷數據與復活方式見 `archive/config_archived_strategies.py` docstring；本文件僅摘要結論。

#### 目前啟用策略（`strategies/config.py` `strategies_raw_all`，共 10 個）

| # | 策略名稱 | 形成期模組 | 交易期模組 | 角色 |
| :---: | :--- | :--- | :--- | :--- |
| 1 | SSD Rolling | `ssd_rolling.py` | `zscore_trading.py` | SSD 家族基準 |
| 2 | DTW Paper Fixed (DTW) | 借用 #DTW 原版配對 | `zscore_trading.py`（路徑 B） | 誠實 DTW 基準 |
| 3 | SSD-DTW-PCA Paper Fixed | 借用 #SSD-DTW-PCA 原版配對 | `zscore_trading.py`（路徑 B） | 全組最佳誠實基準（Sharpe 0.56） |
| 4 | HDBSCAN Cluster SSD-DTW-PCA | `HDBSCAN_Cluster_SSD_DTW.py` | `zscore_trading.py` | 分組消融：HDBSCAN vs GICS（對照 #3） |
| 5 | SSD Rolling DRL THR | 借用 #1 配對 | `drl_threshold_trading.py` | DRL 疊加對照組 |
| 6 | HDBSCAN Cluster SSD-DTW-PCA DRL THR | 借用 #4 配對 | `drl_threshold_trading.py` | DRL 疊加實驗組 |
| 7 | HDBSCAN Cluster SSD-DTW-PCA PCA5 | `HDBSCAN_Cluster_SSD_DTW.py`（5 維 PCA） | `zscore_trading.py` | 維度詛咒修復版（對照 #4） |
| 8 | HDBSCAN Cluster SSD-DTW-PCA PCA5 DRL THR | 借用 #7 配對 | `drl_threshold_trading.py` | 全組 DRL 疊加次高 Sharpe（0.54） |
| 9 | Agglomerative Fundamentals | `agglomerative_fundamentals.py` | `zscore_trading.py` | 分組消融第三支：價格 PCA ⊕ 公司基本面 |
| 10 | Agglomerative Fundamentals DRL THR | 借用 #9 配對 | `drl_threshold_trading.py` | 全組最佳年化報酬（3.03%） |

`formation_strategy_id_base` 機制：#2/#3/#5/#6/#8/#10 借用其他策略已算好的形成期配對（不重複計算），只在交易期套用不同的交易邏輯或座標修正。

#### 📂 現役形成期模組檔案

- `ssd_rolling.py` — 對數價格 Z-Score 標準化 SSD，OLS 對沖比例（#1 直接使用；#9 內部呼叫其排序流程）
- `DTW_Cointegration_Paper.py` — 共整合篩選 + Sakoe-Chiba DTW + PCA 融合（#2/#3 借用之原始配對；#4/#7 組合使用其排序邏輯）
- `HDBSCAN_PCA_Loadings.py` — 報酬 PCA 因子載荷特徵萃取（被 #4/#7/#9 組合複用，見下）
- `HDBSCAN_Cluster_SSD_DTW.py` — 組合 `HDBSCAN_PCA_Loadings`（聚類）+ `DTW_Cointegration_Paper`（排序），#4/#7 使用
- `agglomerative_fundamentals.py` — 組合 `HDBSCAN_PCA_Loadings`（價格特徵）+ 基本面 + Agglomerative 分群 + `ssd_rolling`（排序），#9 使用
- `_utils.py` — 共用統計工具（`_ols`、`_adf_stat`、`_compute_hurst`），所有模組共用

> **架構模式：「組合優於重寫」**——#4/#7/#9 都不是從零打造的獨立形成邏輯，而是把既有模組（`HDBSCAN_PCA_Loadings` 的特徵萃取、`DTW_Cointegration_Paper`／`ssd_rolling` 的共整合篩選與排序）像積木一樣組裝，只替換「配對候選怎麼分組」這一個環節。這讓每個新策略都是乾淨的單變因消融實驗（ablation），而非引入多個同時變動的變因。

#### 統一三道統計過濾（所有現役策略共用，經 `_utils.py` 實作）

| 過濾條件 | 公式 / 判定 | SSD Rolling | DTW Paper | HDBSCAN 系列 | Agglomerative |
| :--- | :--- | :---: | :---: | :---: | :---: |
| ADF 共整合 p 值上限 | $p <$ threshold | 0.05 | 0.01 | 0.01 | 0.05 |
| OU 半衰期範圍 | $1 \le HL \le T_{trading}/3$ 天 | ✓ | ✓ | ✓ | ✓ |
| Hurst 指數上限 | $H < 0.50$ | ✓ | ✓ | ✓ | ✓ |

> **ADF 使用 `regression="n"`**（無截距無趨勢），相比 Engle-Granger 標準程序（`"c"`）更保守。
> **OU 半衰期** 計算：$\lambda$ 為 AR(1) 係數，$HL = -\ln(2)/\lambda$，$\lambda < 0$ 才具均值回歸性。
> **Hurst 指數** 使用 R/S 分析，對殘差序列直接計算（`already_stationary=True`，不再做一次差分）。

## 📏 一、 SSD Rolling 形成期邏輯 (`ssd_rolling.py`)

SSD 家族目前唯一現役成員（`ssd_basic.py` 累積回報比值版已封存，見文末）。以同 GICS 產業內、Z-Score 標準化對數價格的歐氏距離平方和篩選配對。

### 1.1 對數價格 Z-Score 標準化

$$P'_{i,t} = \frac{\ln P_{i,t} - \mu_{\ln P_i}}{\sigma_{\ln P_i}}$$

使不同絕對價格水準的股票具有相同的波動度尺度。

### 1.2 協方差矩陣 OLS 避險比例（批次計算，外層 $i$=Ticker_B=X，內層 $j$=Ticker_A=Y）

$$\beta = \frac{\text{Cov}(P'_A,\, P'_B)}{\text{Var}(P'_B)}, \quad \epsilon_t = P'_{A,t} - \beta \cdot P'_{B,t}$$

先按 SSD 初篩取前 $\max(200, N_{top} \times 15)$ 候選對，再對候選對做慢速統計計算（ADF/半衰期/Hurst），大幅減少計算量。

### 1.3 三道統計過濾（同產業內）

沿用上方統一三道過濾（ADF $p<0.05$、半衰期 $1$–$T_{trading}/3$ 天、Hurst $<0.50$），通過後依 SSD 升序取前 `top_n`。

### 存入 `Formation_Params` 的欄位

| 欄位 | 說明 |
| :--- | :--- |
| `Hedge_Ratio` | 形成期 OLS 斜率 $\beta$ |
| `Spread_Mean` / `Spread_Std` | $\mu_\epsilon$ / $\sigma_\epsilon$ |
| `Log_Mean_A/B`, `Log_Std_A/B` | 形成期對數價格均值/標準差（Z-Score 中心化與縮放用） |

> **與已封存 SSD Basic 的差異**：Basic 用累積回報指數（價格比值空間）、固定 $\beta=1$；Rolling 用 Z-Score 對數空間、估計 OLS $\beta$。Rolling 是 Basic 的滾動延伸版，兩者交易期 spread 重建路徑因此分叉（見 `notebooks/trading.ipynb` 路徑 B1/B2）。

### 1.4 SSD Rolling 在組合模式中的第二重角色

`ssd_rolling.Formation` 不只被 #1 直接呼叫，也被 `agglomerative_fundamentals.py`（#9）在分群完成後複用：把 Agglomerative 的分群標籤當作 `sector_mapping` 餵給它，**完全沿用其 min-SSD 排序 + 共整合篩選流程**，只是分組依據從真實 GICS 換成資料驅動的分群結果。

## ⏳ 二、 DTW Cointegration Paper 形成期邏輯 (`DTW_Cointegration_Paper.py`)

動態時間扭曲（DTW）容忍兩支股票走勢存在**時間滯後**的情況，比純 SSD 歐氏距離更具彈性。現役策略 #2（DTW Paper Fixed）與 #3（SSD-DTW-PCA Paper Fixed）**借用**本模組原版計算的形成期配對（`formation_strategy_id_base`），僅在交易期做座標修正（見下方「Paper Fixed」說明）；#4/#7 則組合本模組的排序邏輯與 HDBSCAN 分組。

### 2.1 對數價格 Z-Score 標準化（同 SSD Rolling）

### 2.2 雙向 OLS 共整合篩選（取 ADF p 值較小的方向）

對每對股票分別做兩個方向的 OLS 回歸，選擇 ADF p 值較小（共整合性更強）的方向作為 $(\text{Ticker\_A},\, \text{Ticker\_B})$。

**四道統計過濾**（$p < 0.01$，比 SSD 更嚴格）：ADF 共整合 → OU 半衰期 → Hurst 指數 → 通過後才計算 SSD/DTW 距離（節省計算）。

### 2.3 Sakoe-Chiba 限制窗口 DTW 距離

$$\text{DTW}_{A,B} = \min_{\text{path}} \sum_{(i,j) \in \text{path}} (P'_{A,i} - P'_{B,j})^2, \quad |i-j| \le W$$

限制時間扭曲在 $W=15$ 天內，防止非理性的長距離時間對齊。

### 2.4 排序模式

| 模式 | 說明 | 對應現役策略 |
| :--- | :--- | :--- |
| `method="dtw"` | 依 DTW 距離升序排序 | #2 DTW Paper Fixed |
| `method="ssd_dtw_pca"` | SSD 與 DTW 標準化後 PCA 取第一主成分升序排序；若 PC1 loadings 方向為負則取反確保「距離越小得分越小」 | #3 SSD-DTW-PCA Paper Fixed（全組最佳誠實基準）、#4/#7（組合 HDBSCAN 分組使用） |

### 存入 `Formation_Params` 的欄位

`Hedge_Ratio`、`OLS_Alpha`（最佳方向 OLS 截距，交易期重建 spread 必需）、`Spread_Mean/Std`（OLS 殘差統計量）、`Log_Mean_A/B`、`Log_Std_A/B`。

### 2.5 「Paper Fixed」座標修正說明（重要：已封存原版 vs 現役 Fixed 版的差異）

原版 DTW Paper（已封存）在標準化空間擬合 OLS，但同時輸出 `OLS_Alpha`，導致交易期 `zscore_trading.py` 誤判為路徑 A（原始 log-price 空間）重建 spread，造成**恆定的 Z-Score 偏移量**（進場 Z 中位數高達 3.24、85% 的交易在期初 3 天內就進場、99% 靠期末強制平倉出場——這是座標系錯位的 artifact，不是真實訊號）。診斷後原版的中位 Sharpe 0.45–0.46 被證實是這個 bug 的副產物；修正後（Fixed 版，`ignore_ols_alpha=True` 強制走路徑 B）中位 Sharpe 跌至 ≈0，但**最佳 Sharpe（Top3, SL0%）反而達到全組最高的 0.56**——代表訊號本身在特定參數組合下是有效的，只是原版的「有效」評估被 bug 污染了。

⚠️ 原版的形成期配對目前仍被 #2/#3 借用（`formation_strategy_id_base`），`formation_data` 中的資料列不可刪除。

## 🌐 三、 HDBSCAN Cluster SSD-DTW-PCA 形成期邏輯 (`HDBSCAN_Cluster_SSD_DTW.py`)

**命題**：「HDBSCAN 資料驅動聚類」是否優於「GICS 靜態產業分類」作為配對搜尋空間？

**實驗設計（控制變因）**：對照組 = #3 SSD-DTW-PCA Paper Fixed（GICS 產業分組 + SSD-DTW-PCA 排序）；實驗組 = 本策略（HDBSCAN 聚類分組 + **完全相同**的 SSD-DTW-PCA 排序 + 相同交易端路徑 B）。唯一差異是分組方式。

### 3.1 組合實作：兩個既有模組串接

1. `HDBSCAN_PCA_Loadings.Formation._build_feature_matrix()` → 報酬 PCA 因子載荷 + HDBSCAN 聚類，產生 cluster labels
2. `DTW_Cointegration_Paper.Formation` → 把 cluster labels 當 `sector_mapping` 傳入，完整沿用其標準化空間 OLS/ADF 篩選、SSD/DTW 距離計算與 PCA 融合排序

HDBSCAN 噪音點（`label=-1`）映射為 `"Unknown"`，DTW 流程自動跳過——聚類的離群過濾能力也因此被帶入。

### 3.2 報酬 PCA 因子載荷特徵（`HDBSCAN_PCA_Loadings.py`）

動機（消融實驗）：統計特徵描述「單股行為像不像」，但與「兩檔股票價格路徑能否共整合」沒有因果關聯。文獻標準做法是以**報酬率的共同因子暴露**作為聚類座標（Avellaneda & Lee 2010 的 eigenportfolios；Sarmento & Horta 2020 的 PCA 降維 + 聚類）：暴露在相同風險因子的股票，才有經濟理由維持長期均衡關係——這正是共整合配對的來源。

$$R \in \mathbb{R}^{(T-1)\times N}\ (\text{日報酬矩陣，逐股標準化}) \;\xrightarrow{\text{PCA}}\; \text{loadings}_i = \text{components}_{:,i} \times \sqrt{\text{explained\_variance}}$$

以 $\sqrt{\text{特徵值}}$ 加權保留因子重要性排序；`reduce_method="none"` 讓 loadings 直接餵給 HDBSCAN（跳過 UMAP，消除滾動窗口間嵌入不穩定的問題——目前所有現役策略皆使用 `reduce_method="none"`）。

### 3.3 維度詛咒診斷與 PCA5 修復（策略 #7）

15 維 PCA loadings（策略 #4 原始設定）平均僅解釋 **58.6%** 報酬變異，後段主成分訊號弱、卻仍等權參與歐氏距離計算，稀釋了密度估計——實測平均 **30.9%**（中位數 35.8%）標的被 HDBSCAN 判為雜訊排除，群數在 2–25 之間劇烈震盪。降至 **5 維**（策略 #7）後：

| 指標 | 15 維（#4） | 5 維（#7） |
| :--- | :---: | :---: |
| 雜訊比例中位數 | 35.8% | 8.1% |
| 群數標準差 | 6.4 | 4.3 |
| Z-Score 正 Sharpe 比例 | 60% | 73% |
| Z-Score 中位數 Sharpe | 0.07 | 0.13 |
| Z-Score 最佳 Sharpe | 0.22 | 0.29 |
| Z-Score 最佳年化報酬 | 0.82% | 1.08% |

全面優於 15 維版本，證實維度詛咒假說。#7 的形成期配對再供 #8（DRL THR 疊加）借用。

### 存入 `Formation_Params` 的欄位

與 DTW Paper 相同：`Hedge_Ratio`、`OLS_Alpha`、`Spread_Mean/Std`、`Log_Mean_A/B`、`Log_Std_A/B`。另外保留 `Sector_A/B`（回填真實 GICS 產業，供 MSR 產業分散與結果分析用；`Sector` 欄位本身仍是群集標籤）。

## 🧬 四、 Agglomerative Fundamentals 形成期邏輯 (`agglomerative_fundamentals.py`)

分組消融實驗的第三支：以「報酬 PCA 因子載荷（價格行為）⊕ GICS 產業 one-hot ⊕ log(市值) ⊕ 盈餘殖利率 $1/PE$（公司基本面）」混合特徵空間做 **Agglomerative Clustering** 分組，取代單純 GICS 靜態分組或純價格導向的 HDBSCAN 分組。

### 4.1 為何用 Agglomerative 而非 HDBSCAN/DBSCAN

密度式分群（HDBSCAN/DBSCAN）在股票特徵空間上容易產生「單一巨型群 + 大量雜訊點」的極端不平衡分布（見上節：HDBSCAN 系列平均 30%+ 標的被判為雜訊排除）。Agglomerative 對每個點都會分配群組（無「雜訊」概念），且改用 **`distance_threshold`**（依每期合併距離的分位數校準）而非固定 `n_clusters`，避免「強迫合併到固定群數」重現同樣的不平衡問題。

### 4.2 混合特徵矩陣建構（區塊分別標準化 + 加權）

1. 借用 `HDBSCAN_PCA_Loadings._build_feature_matrix()` 取得報酬 PCA 因子載荷（`reduce_method="none"`），**不使用**其 HDBSCAN 聚類本身
2. 合併基本面特徵：`log1p(市值)`、`1/本益比`（盈餘殖利率），依產業中位數插補缺失值後 winsorize（1%–99%）
3. 三個區塊（價格 PCA、基本面、GICS one-hot）**各自** `StandardScaler` 標準化後依權重拼接：

$$X = [\, w_{price} \cdot \text{Scale}(\text{PCA loadings}) \;\|\; w_{fund} \cdot \text{Scale}([\log\text{MktCap},\, 1/PE]) \;\|\; w_{sector} \cdot \text{OneHot(GICS)} \,]$$

> **避免的已知失敗模式**：單一 joint `StandardScaler` 對整個拼接矩陣做一次標準化，會讓 one-hot 欄位數量稀釋連續特徵的距離量測——這正是本專案先前已證偽的「特徵未加權原始拼接」失敗模式（見 HDBSCAN 舊特徵系封存記錄）。區塊分別標準化 + 顯式權重是本模組刻意避開此陷阱的設計。

### 4.3 自適應 `distance_threshold` 校準

$$\text{threshold} = \text{percentile}_{75}(\{\text{合併距離}\}_{\text{完整 dendrogram probe}})$$

先用 `n_clusters=1` 跑一次完整合併路徑取得所有合併距離，再以 75th 分位數作為最終 `AgglomerativeClustering(distance_threshold=...)` 的門檻——依每期資料分布動態校準，不使用固定絕對值。過小群併入 `"Unknown"`（排除）。

### 4.4 分群結果接回既有排序流程

分群標籤當作 `sector_mapping` 餵給既有 `ssd_rolling.Formation`，**完全複用**其 min-SSD 排序 + Engle-Granger ADF + 半衰期 + Hurst 篩選流程（見一、SSD Rolling 章節），交易端沿用既有 Z-Score / Beta 動態配重 / 停損引擎，未做任何修改——確保這是乾淨的單變因消融（只換分組方式）。

### 4.5 已知且刻意接受的限制：基本面前視偏誤

基本面資料為**單一時點靜態快照**（yfinance 今日資料，見 `fetch/fundamentals_yfinance.py`），非 2000–2025 逐點歷史資料——免費資料源無法取得歷史逐日基本面（需 WRDS/Compustat 等付費資料商）。這代表每個歷史形成窗口都使用同一組「現在的」市值/本益比，對早期窗口存在前視偏誤，是已知且刻意接受的限制。

### 存入的額外欄位

`Sector_A/B`（真實 GICS）、`Cluster_ID_A/B`（分群標籤）、`MarketCap_A/B`、`TrailingPE_A/B`，供報表與後續分析使用；`Formation_Params` 核心欄位與 SSD Rolling 相同。

### 4.6 目前最佳「ML 配對」結果

Z-Score 基準（#9）：最佳 Sharpe 0.35、最佳年化報酬 2.70%，雙指標都優於 SSD (Rolling) 的 0.26 / 2.30%，是目前所有分組消融實驗中唯一在兩個指標上都超越 SSD 基準的策略。疊加 DRL-THR 後（#10）最佳年化報酬進一步提升至全組最高的 **3.03%**（詳見 `notebooks/trading.ipynb` 績效比較章節）。

In [ ]:
"""
Agglomerative Fundamentals — 自適應 distance_threshold 校準示例
完整重現 _cluster() 的邏輯，用於驗證分位數校準機制。
"""
import numpy as np
from sklearn.cluster import AgglomerativeClustering

rng = np.random.default_rng(42)
# 模擬一個形成期的混合特徵矩陣（20 檔股票 × 8 維：5 價格 PCA + 2 基本面 + 1 簡化 one-hot）
X = rng.normal(size=(20, 8))

# 步驟 1：完整合併路徑 probe（n_clusters=1 強迫合併到底，取得所有合併距離）
probe = AgglomerativeClustering(n_clusters=1, linkage="average", compute_distances=True).fit(X)
distances = probe.distances_
threshold = float(np.percentile(distances, 75.0))
print(f"合併距離範圍：[{distances.min():.3f}, {distances.max():.3f}]")
print(f"75th 分位數 distance_threshold = {threshold:.3f}")

# 步驟 2：用校準出的門檻做最終分群（distance_threshold 與 n_clusters 互斥，故 n_clusters=None）
final = AgglomerativeClustering(n_clusters=None, distance_threshold=threshold, linkage="average").fit(X)
n_clusters = len(set(final.labels_))
sizes = np.bincount(final.labels_)
print(f"最終群數：{n_clusters}，各群大小：{sorted(sizes.tolist(), reverse=True)}")
print("→ 與 HDBSCAN 不同：每個點都被分配到某個群，無「雜訊」標籤")


## 📖 五、 已封存策略摘要

以下策略已從 `strategies_raw_all` 移出至 `archive/config_archived_strategies.py`，歷史回測結果保留於 `results/result.db`，完整診斷數據、失敗根因分析與復活方式見該檔案 docstring。此處僅列封存結論。

| 分類 | 策略 | 封存原因摘要 |
| :--- | :--- | :--- |
| 座標系 artifact | DTW Paper 原版 ×2 | OLS 標準化空間擬合但輸出 `OLS_Alpha`，交易端誤判路徑 A → 恆定 Z 偏移；形成期配對仍被 Fixed 版借用 |
| 舊特徵系 | HDBSCAN MultiScale/UMAP ×4 | 10 維單股統計特徵與共整合無因果關聯，Quality Score 排序不顯著（中位 Sharpe −0.06~+0.10） |
| Ensemble | Ensemble HDBSCAN/SSD-DTW ×2 | 交集/聯集邏輯放大劣質配對（中位 Sharpe −0.24，比子策略更差） |
| SSD Basic | SSD Basic | 被 SSD Rolling（滾動延伸版）取代 |
| Kalman | SSD Rolling/HDBSCAN UMAP Kalman ×2 | 與論文兩大命題（ML 配對、DRL 交易）無關；年化報酬僅 0.3%/0.03% |
| CONV 收斂持有 | DTW/HDBSCAN CONV ×2 | 長持收斂假設經檢驗被否決（Sharpe −0.11/−0.33） |
| DRL v1/v3 | DRL LSTM ×3、DRL FQI ×3 | 逐日定位動作空間已系統性證偽（OOS 過度交易，中位 Sharpe −0.71~−2.30），由 v4 門檻選擇式取代 |
| HDBSCAN PCA-Loadings 系列 | HDBSCAN PCA-Loadings（Z-Score/DRL THR） | 15 維 PCA loadings 平均解釋 58.6% 變異、雜訊比例 30.9%，被 Cluster-SSD-DTW-PCA 系列（含 PCA5 修復版）全面取代 |
| ML Pair Quality | ML Pair Quality | 監督式學習排序（walk-forward 反事實回歸）兩次嘗試（原始標籤、log 壓縮標籤）皆未產生可學習訊號，15 組 Z-Score 對照全負或接近零 Sharpe |

**從未進入策略清單的孤兒模組**：`archive/trading/pure_dtw_trading.py`（早期交叉回穿波帶構想）與 `archive/trading/drl_lstm_v2_trading.py`（DRL v1→v3 演進鏈的中繼修復版）從未被賦予正式 config 條目，`result.db` 無對應回測數據，程式碼保留供架構脈絡參考。

## 📊 六、 現役策略形成期對比總表

### 6.1 分組方式與 Spread 空間對比

| 策略 | 分組依據 | Spread 空間（`Formation_Params`） | 關鍵欄位 |
| :--- | :--- | :--- | :--- |
| SSD Rolling (#1) | 真實 GICS 產業 | Z-Score 標準化 log-price | `Log_Mean/Std_A/B` |
| DTW Paper Fixed (#2/#3) | 真實 GICS 產業（借用原版配對） | Z-Score 標準化 log-price（路徑 B，強制忽略 `OLS_Alpha`） | `OLS_Alpha`（存在但交易端忽略）、`Log_Mean/Std_A/B` |
| HDBSCAN Cluster SSD-DTW-PCA (#4/#7) | HDBSCAN 聚類（報酬 PCA 因子載荷） | 同上（路徑 B） | 同上 + `Sector_A/B`（真實 GICS 回填） |
| Agglomerative Fundamentals (#9) | Agglomerative 聚類（價格 PCA ⊕ 基本面 ⊕ GICS one-hot） | Z-Score 標準化 log-price（SSD Rolling 排序） | `Log_Mean/Std_A/B` + `Cluster_ID`、`MarketCap`、`TrailingPE` |

### 6.2 交易期 Spread 重建路徑（由 `Formation_Params` 決定，詳見 `notebooks/trading.ipynb`）

```
現役策略全數透過 ignore_ols_alpha=True 或未輸出 OLS_Alpha 走「路徑 B」：
  spread = Z-Score(ln P_A) - beta * Z-Score(ln P_B)
  → 適用於 SSD Rolling、DTW Paper Fixed、HDBSCAN 系列、Agglomerative Fundamentals

（路徑 A：原始 log-price OLS 殘差空間，僅原版 DTW/HDBSCAN 舊特徵系〔已封存〕使用）
```

> 現役策略統一使用路徑 B，確保跨策略的交易端計算完全一致，任何績效差異只能來自形成期配對品質本身——這是分組消融實驗能夠乾淨比較的關鍵前提。

## 📖 參考文獻

本研究各形成期策略直接引自以下論著：

---

### Gatev, Goetzmann & Rouwenhorst (2006) — SSD 距離法基礎

> **論著：** Gatev, E., Goetzmann, W. N., & Rouwenhorst, K. G. (2006). Pairs trading: Performance of a relative value arbitrage rule. *Review of Financial Studies*, **19**(3), 797–827.

**論著核心貢獻：**

- 提出以**累積回報指數（normalized price index）**作為配對距離計算基礎，消除絕對價格差距
- 採用**歐氏距離平方和（SSD）**作為相似度量度，計算直觀且效率高
- 固定等市值對沖比例，實現美元中性策略，消除市場 Beta 暴露
- 1962–2002 年 S&P 500 資料，年化超額報酬約 6%（交易成本前），驗證統計套利可行性

**本研究對應：** SSD 距離概念是 `ssd_rolling.py` 的理論起點（已封存的 `ssd_basic.py` 更直接對應原始累積回報指數版本）

---

### Engle & Granger (1987) — 共整合統計理論

> **論著：** Engle, R. F., & Granger, C. W. J. (1987). Co-integration and error correction: Representation, estimation, and testing. *Econometrica*, **55**(2), 251–276.

**論著核心貢獻：**

- 提出**共整合（cointegration）**的正式定義：兩個 I(1) 序列的線性組合為 I(0) 則稱共整合
- ADF 單根檢定作為共整合殘差定常性的標準驗證方法

**本研究對應：** 所有現役策略的 ADF 共整合過濾（`_utils.py` 之 `_adf_stat`）

---

### Krauss, Do & Huck (2016) — 三道統計過濾的學術依據

> **論著：** Krauss, C., Do, X. A., & Huck, N. (2016). The profitability of pairs trading strategies: Distance, cointegration and copula methods. *European Journal of Operational Research*.

**論著核心貢獻：**

- 系統比較距離法、共整合法、Copula 法三類配對選取方法；共整合法在風險調整後報酬上優於純距離法
- **Hurst 指數**（$H<0.5$）輔助過濾驗證殘差均值回歸性；**OU 半衰期**確保交易期內有足夠的均值回歸次數

**本研究對應：** 三道統計過濾（所有現役策略共用）：ADF $p<0.01$–$0.05$、$1 \le HL \le T_{trading}/3$、$H<0.50$

---

### Zhu (2024) — DTW 形成期策略來源

> **論著：** Zhu, M. (2024). Pairs trading with dynamic time warping. Working Paper.

**論著核心貢獻：**

- 引入**動態時間扭曲（DTW）**度量股票走勢相似度，容忍時間滯後（lead-lag effect）
- Sakoe-Chiba 帶約束（$W=15$ 天）防止不合理的長距離時間對齊
- SSD 與 DTW 距離的 PCA 融合取第一主成分作為融合排序依據

**本研究對應：** `DTW_Cointegration_Paper.py`（策略 #2/#3 借用其原始配對；#4/#7 組合複用其排序邏輯）

---

### Campello, Moulavi & Sander (2013) — HDBSCAN 聚類演算法

> **論著：** Campello, R. J., Moulavi, D., & Sander, J. (2013). Density-based clustering based on hierarchical density estimates. *PAKDD*.

**論著核心貢獻：**

- 提出**層次密度分群（HDBSCAN）**：自動確定群數，無需預先指定 $k$
- 噪音點（outliers）標記為 `label=-1`，適合過濾行為異常的股票

**本研究對應：** `HDBSCAN_Cluster_SSD_DTW.py`（策略 #4/#7 的分組步驟）

---

### Avellaneda & Lee (2010) — PCA 因子載荷 / Eigenportfolios

> **論著：** Avellaneda, M., & Lee, J.-H. (2010). Statistical arbitrage in the U.S. equities market. *Quantitative Finance*, **10**(7), 761–782.

**論著核心貢獻：**

- 以 PCA 從報酬率中萃取共同風險因子（eigenportfolios），股票報酬 = 因子暴露 + 特異報酬（idiosyncratic return）
- 特異報酬以 OU 過程建模，驗證均值回歸的統計套利可行性
- 1997–2007 年回測，PCA 策略平均年化 Sharpe 達 1.44

**本研究對應：** `HDBSCAN_PCA_Loadings.py` 的報酬 PCA 因子載荷特徵萃取（被 #4/#7/#9 組合複用），取代單股統計特徵作為聚類座標

---

### Sarmento & Horta (2020) — PCA 降維 + 聚類的配對搜尋方法論

> **論著：** Sarmento, S. M., & Horta, N. (2020). Enhancing a pairs trading strategy with the application of machine learning. *Expert Systems with Applications*, **158**, 113490.

**論著核心貢獻：**

- 提出「降維 → 無監督聚類 → 規則化配對篩選」三段式配對搜尋框架
- 以 PCA 降維後的報酬表徵搭配 OPTICS/DBSCAN 分群，取代傳統產業分類作為配對搜尋空間

**本研究對應：** `HDBSCAN_Cluster_SSD_DTW.py`／`agglomerative_fundamentals.py` 共用的「先降維/萃取特徵、再聚類分組、再套用既有排序規則」整體架構模式

---

### Ward (1963) — 階層聚類（Agglomerative Clustering）方法論

> **論著：** Ward, J. H. (1963). Hierarchical grouping to optimize an objective function. *Journal of the American Statistical Association*, **58**(301), 236–244.

**論著核心貢獻：**

- 提出階層式聚集聚類（agglomerative hierarchical clustering）的目標函數框架：逐步合併使組內變異增量最小的群集
- 相較 K-Means 等分割式方法，不需預先指定群數，且產生的合併順序（dendrogram）可用於事後校準任意分群粒度

**本研究對應：** `agglomerative_fundamentals.py` 的 `AgglomerativeClustering`（average linkage）分組步驟，以及依合併距離分位數動態校準 `distance_threshold` 的設計

---

### Hong & Hwang (2021) — 公司基本面驅動的配對搜尋

> **論著：** Hong, S., & Hwang, S. (2021). In search of pairs using firm fundamentals: Is pairs trading profitable? *The European Journal of Finance*, **29**(5).

**論著核心貢獻：**

- 主張以**企業基本面特徵**（而非純價格序列）識別潛在配對，能捕捉價格序列無法反映的長期均衡關係來源
- 基本面驅動的配對策略在納入交易成本後仍展現統計上顯著的獲利能力

**本研究對應：** `agglomerative_fundamentals.py` 混合特徵中的 `log(市值)` 與盈餘殖利率 $(1/PE)$ 區塊，是本研究唯一同時在 Sharpe 與年化報酬兩項指標上超越 SSD 基準的分組方式（策略 #9/#10）

> ⚠️ **勘誤**：本文件先前版本誤將此文獻標註為 "Han, He, Rapach & Zhou (2021)"——該作者組合實際對應另一篇不相關文獻（Cross-Sectional Expected Returns 相關的 Fama-MacBeth 迴歸研究）。本次更新已修正為正確作者 Hong 與 Hwang。